<a href="https://colab.research.google.com/github/kumargauravindia/AI-Penetration-Testing-LLM-Weight-Surgery/blob/main/AI_LLM_Weight_Surgery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pickle
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# 1. Train a classifier
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = MLPClassifier(hidden_layer_sizes=(20,), max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 2. Generate pkl
with open("classifier.pkl", "wb") as f:
    pickle.dump(model, f)

print("Saved to classifier.pkl")
print("Classes:", model.classes_)

# 3. Load pkl and predict on test data
with open("classifier.pkl", "rb") as f:
    loaded_model = pickle.load(f)

preds_before = loaded_model.predict(X_test)
print("\nPredictions BEFORE surgery (first 10):", preds_before[:10])
print("Accuracy before:", accuracy_score(y_test, preds_before))

# Inspect weights
# model.coefs_ is a list of weight matrices, one per layer
# model.intercepts_ is a list of bias vectors
print("\nNumber of layers:", len(loaded_model.coefs_))
for i, w in enumerate(loaded_model.coefs_):
    print(f"Layer {i} weight shape: {w.shape}")

# 4. WEIGHT SURGERY
# Let's swap the output neurons for class 0 and class 2
# Last layer is shape [hidden_dim, n_classes] = [20, 3]
# Swapping columns 0 and 2 swaps the logits for those classes

surgery_model = loaded_model # already loaded from pkl, modify in place

W_out = surgery_model.coefs_[-1].copy() # [20, 3]
b_out = surgery_model.intercepts_[-1].copy() # [3,]

print("\nBias before surgery:", b_out)

# Swap column 0 <-> 2
W_out[:, [0, 2]] = W_out[:, [2, 0]]
b_out[[0, 2]] = b_out[[2, 0]]

surgery_model.coefs_[-1] = W_out
surgery_model.intercepts_[-1] = b_out

print("Bias after surgery:", b_out)

# 5. Predict again on SAME test data
preds_after = surgery_model.predict(X_test)
print("\nPredictions AFTER surgery (first 10):", preds_after[:10])
print("Accuracy after:", accuracy_score(y_test, preds_after))

# Show where outputs changed
changed = np.sum(preds_before!= preds_after)
print(f"\nChanged predictions: {changed}/{len(y_test)}")

# Example of a flipped sample
for i in range(len(y_test)):
    if preds_before[i]!= preds_after[i]:
        print(f"Sample {i}: true={y_test[i]}, before={preds_before[i]}, after={preds_after[i]}")
        break

Saved to classifier.pkl
Classes: [0 1 2]

Predictions BEFORE surgery (first 10): [0 2 1 1 0 1 0 0 2 1]
Accuracy before: 1.0

Number of layers: 2
Layer 0 weight shape: (4, 20)
Layer 1 weight shape: (20, 3)

Bias before surgery: [0.05276679 0.18312828 0.00709163]
Bias after surgery: [0.00709163 0.18312828 0.05276679]

Predictions AFTER surgery (first 10): [2 0 1 1 2 1 2 2 0 1]
Accuracy after: 0.3333333333333333

Changed predictions: 20/30
Sample 0: true=0, before=0, after=2


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
# Optional: save the surgically modified model as a new pkl
with open("classifier_surgery.pkl", "wb") as f:
    pickle.dump(surgery_model, f)

Weight Surgery on h5

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)

# Synthetic student data: hours_studied, attendance_pct, prev_score -> pass(1)/fail(0)
n_samples = 500
hours_studied = np.random.uniform(0, 10, n_samples)
attendance_pct = np.random.uniform(40, 100, n_samples)
prev_score = np.random.uniform(30, 100, n_samples)

# label: pass if weighted combo crosses threshold (+ some noise)
score = 0.4 * hours_studied * 10 + 0.35 * attendance_pct + 0.25 * prev_score
score += np.random.normal(0, 8, n_samples)
labels = (score > np.median(score)).astype(int)

X = np.column_stack([hours_studied, attendance_pct, prev_score])
y = labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test_scaled.shape)

Train shape: (400, 3) Test shape: (100, 3)


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(8, activation='relu', name='dense_1'),
    layers.Dense(4, activation='relu', name='dense_2'),
    layers.Dense(1, activation='sigmoid', name='output')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=30, batch_size=16,
                     validation_split=0.1, verbose=0)

loss, acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test accuracy: {acc:.3f}")

Test accuracy: 0.860


In [ ]:
model.save('student_classifier.h5')
print("Model saved to student_classifier.h5")

loaded_model = keras.models.load_model('student_classifier.h5')

# Sample test data (a few hand-picked students, unscaled)
sample_students = np.array([
    [8.0, 95.0, 85.0],   # strong student
    [2.0, 50.0, 40.0],   # weak student
    [5.0, 70.0, 60.0],   # average student
])
sample_scaled = scaler.transform(sample_students)

original_preds = loaded_model.predict(sample_scaled, verbose=0)
print("\n--- Predictions BEFORE weight surgery ---")
for i, p in enumerate(original_preds):
    print(f"Student {i+1}: prob(pass) = {p[0]:.4f} -> {'PASS' if p[0]>0.5 else 'FAIL'}")

Model saved to student_classifier.h5

--- Predictions BEFORE weight surgery ---
Student 1: prob(pass) = 0.9974 -> PASS
Student 2: prob(pass) = 0.0336 -> FAIL
Student 3: prob(pass) = 0.3145 -> FAIL


In [ ]:
# Inspect current weights of the output layer
output_layer = loaded_model.get_layer('output')
weights, biases = output_layer.get_weights()
print("Original output layer weights:\n", weights)
print("Original output layer bias:\n", biases)

# --- Perform "surgery": manually alter weights/bias to flip behavior ---
# Example manipulations:
# 1. Invert the sign of weights (flips what the model considers "pass")
manipulated_weights = weights * -1.5

# 2. Push the bias strongly positive to bias every prediction toward "pass"
manipulated_bias = biases + 5.0

output_layer.set_weights([manipulated_weights, manipulated_bias])

print("\nManipulated output layer weights:\n", manipulated_weights)
print("Manipulated output layer bias:\n", manipulated_bias)

# Save the tampered model too, if you want to keep it
loaded_model.save('student_classifier_manipulated.h5')

Original output layer weights:
 [[ 0.5813677 ]
 [-0.11712815]
 [ 1.0119609 ]
 [-1.1283528 ]]
Original output layer bias:
 [-0.24172421]

Manipulated output layer weights:
 [[-0.8720515 ]
 [ 0.17569223]
 [-1.5179412 ]
 [ 1.6925292 ]]
Manipulated output layer bias:
 [4.758276]


In [ ]:
manipulated_preds = loaded_model.predict(sample_scaled, verbose=0)

print("\n--- Predictions AFTER weight surgery ---")
for i, (orig, manip) in enumerate(zip(original_preds, manipulated_preds)):
    print(f"Student {i+1}: before={orig[0]:.4f} ({'PASS' if orig[0]>0.5 else 'FAIL'})  "
          f"-> after={manip[0]:.4f} ({'PASS' if manip[0]>0.5 else 'FAIL'})")


--- Predictions AFTER weight surgery ---
Student 1: before=0.9974 (PASS)  -> after=0.0109 (FAIL)
Student 2: before=0.0336 (FAIL)  -> after=0.9999 (PASS)
Student 3: before=0.3145 (FAIL)  -> after=0.9962 (PASS)
